In [1]:
"""
Task 4: Loan Default Risk with Business Cost Optimization
=========================================================
Dataset: Home Credit Default Risk (synthetic)
Models:  Logistic Regression, GradientBoosting (CatBoost-style)
Focus:   Cost-benefit threshold optimization, feature importance
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (confusion_matrix, f1_score, roc_curve, auc,
                             classification_report, precision_recall_curve,
                             average_precision_score)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ─────────────────────────────────────────────
# 1. GENERATE HOME CREDIT DATASET
# ─────────────────────────────────────────────
def generate_home_credit(n=10000):
    """Synthetic dataset matching Home Credit schema with ~8% default rate."""
    age          = np.random.normal(43, 11, n).clip(21, 69).astype(int)
    income       = np.random.exponential(150000, n).clip(30000, 1000000)
    loan_amount  = income * np.random.uniform(0.5, 6.0, n)
    loan_amount  = np.clip(loan_amount, 10000, 2000000)
    annuity      = loan_amount / np.random.uniform(24, 96, n)
    credit_ratio = loan_amount / income
    employed_yrs = np.random.exponential(5, n).clip(0, 40)
    gender       = np.random.choice(['M','F'], n, p=[0.36, 0.64])
    education    = np.random.choice(['Secondary','Higher education','Incomplete higher',
                                     'Lower secondary','Academic degree'], n,
                                    p=[0.57, 0.28, 0.08, 0.04, 0.03])
    family_status= np.random.choice(['Married','Single','Civil marriage',
                                     'Separated','Widow'], n, p=[0.63, 0.14, 0.12, 0.07, 0.04])
    house_own    = np.random.choice(['House/apartment','With parents','Municipal apartment',
                                     'Rented apartment','Office apartment'], n,
                                    p=[0.72, 0.11, 0.08, 0.07, 0.02])
    ext_source_1 = np.random.uniform(0.1, 0.9, n)
    ext_source_2 = np.random.uniform(0.0, 1.0, n)
    ext_source_3 = np.random.uniform(0.0, 1.0, n)
    prev_loans   = np.random.poisson(2, n)
    days_id_pub  = np.random.randint(-6000, -100, n)
    region_pop   = np.random.exponential(0.02, n).clip(0.0001, 0.1)
    flag_doc3    = np.random.choice([0, 1], n, p=[0.15, 0.85])
    flag_phone   = np.random.choice([0, 1], n, p=[0.28, 0.72])

    # Simulate default probability
    logit = (
        -3.2
        + 0.8  * credit_ratio
        - 1.5  * ext_source_2
        - 1.2  * ext_source_3
        - 0.5  * ext_source_1
        - 0.02 * employed_yrs
        - 0.01 * (age - 40)
        + 0.3  * (education == 'Lower secondary').astype(float)
        - 0.2  * (education == 'Higher education').astype(float)
        - 0.3  * (house_own == 'House/apartment').astype(float)
        + 0.2  * prev_loans / 5.0
        + np.random.normal(0, 0.8, n)
    )
    prob    = 1 / (1 + np.exp(-logit))
    default = (np.random.random(n) < prob).astype(int)

    df = pd.DataFrame({
        'SK_ID_CURR':         np.arange(100001, 100001 + n),
        'TARGET':             default,
        'AMT_INCOME_TOTAL':   income,
        'AMT_CREDIT':         loan_amount,
        'AMT_ANNUITY':        annuity,
        'CREDIT_INCOME_RATIO':credit_ratio,
        'DAYS_BIRTH':         -age * 365,
        'DAYS_EMPLOYED':      -employed_yrs * 365,
        'CODE_GENDER':        gender,
        'NAME_EDUCATION_TYPE':education,
        'NAME_FAMILY_STATUS': family_status,
        'NAME_HOUSING_TYPE':  house_own,
        'EXT_SOURCE_1':       ext_source_1,
        'EXT_SOURCE_2':       ext_source_2,
        'EXT_SOURCE_3':       ext_source_3,
        'CNT_PREV_LOANS':     prev_loans,
        'DAYS_ID_PUBLISH':    days_id_pub,
        'REGION_POPULATION_RELATIVE': region_pop,
        'FLAG_DOCUMENT_3':    flag_doc3,
        'FLAG_PHONE':         flag_phone,
    })
    return df

df = generate_home_credit(10000)
print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['TARGET'].mean():.3%}")

# ─────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────
drop_cols = ['SK_ID_CURR']
cat_cols  = ['CODE_GENDER','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE']
le = LabelEncoder()
df_enc = df.drop(drop_cols, axis=1).copy()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

# Add engineered features
df_enc['ANNUITY_INCOME_RATIO'] = df_enc['AMT_ANNUITY'] / df_enc['AMT_INCOME_TOTAL']
df_enc['CREDIT_ANNUITY_RATIO'] = df_enc['AMT_CREDIT']  / df_enc['AMT_ANNUITY']
df_enc['AGE_YEARS']            = -df_enc['DAYS_BIRTH'] / 365
df_enc['EMPLOYED_YEARS']       = (-df_enc['DAYS_EMPLOYED']).clip(0) / 365
df_enc['EXT_MEAN']             = df_enc[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)

X = df_enc.drop('TARGET', axis=1)
y = df_enc['TARGET']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Test default rate: {y_test.mean():.3%}")

# ─────────────────────────────────────────────
# 3. TRAIN MODELS
# ─────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', C=0.1, random_state=42),
    'GradientBoosting':    GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                      learning_rate=0.05, subsample=0.8,
                                                      random_state=42),
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_tr_sc, y_train)
        y_proba = model.predict_proba(X_te_sc)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]

    fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    y_pred  = (y_proba >= 0.5).astype(int)
    f1      = f1_score(y_test, y_pred)
    results[name] = {'model': model, 'y_proba': y_proba, 'fpr': fpr, 'tpr': tpr,
                     'auc': roc_auc, 'thresholds': thresholds_roc, 'f1': f1}
    print(f"\n{name}: AUC={roc_auc:.4f}  F1={f1:.4f}")

# ─────────────────────────────────────────────
# 4. COST-BENEFIT THRESHOLD OPTIMIZATION
# ─────────────────────────────────────────────
# Business costs:
# FP (approve defaulter)   → bank loses entire loan (HIGH cost)
# FN (reject good customer) → bank loses profit on loan
# TP (reject defaulter)    → bank saves default loss
# TN (approve good)        → bank earns interest

LOAN_DEFAULT_LOSS  = 50000   # Expected loss on a defaulted loan ($)
PROFIT_PER_LOAN    = 5000    # Expected profit on a good loan ($)
COST_FP            = LOAN_DEFAULT_LOSS   # Approve a defaulter
COST_FN            = PROFIT_PER_LOAN     # Reject a good customer

def compute_business_cost(y_true, y_proba, threshold, cost_fp, cost_fn):
    y_pred = (y_proba >= threshold).astype(int)
    cm     = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = fp * cost_fp + fn * cost_fn
    return total_cost, fp, fn, tp, tn

# Sweep thresholds
best_model_name = 'GradientBoosting'
best_proba      = results[best_model_name]['y_proba']
thresholds      = np.linspace(0.01, 0.99, 200)
costs, fps, fns = [], [], []
for t in thresholds:
    cost, fp_c, fn_c, _, _ = compute_business_cost(y_test, best_proba, t, COST_FP, COST_FN)
    costs.append(cost)
    fps.append(fp_c)
    fns.append(fn_c)

opt_idx = np.argmin(costs)
opt_threshold = thresholds[opt_idx]
opt_cost      = costs[opt_idx]
print(f"\n{'='*50}")
print(f"OPTIMAL THRESHOLD: {opt_threshold:.3f}")
print(f"MINIMUM COST:      ${opt_cost:,.0f}")
y_pred_opt = (best_proba >= opt_threshold).astype(int)
cm_opt     = confusion_matrix(y_test, y_pred_opt)
print(f"CM at optimal threshold:\n{cm_opt}")
print(f"F1 at optimal: {f1_score(y_test, y_pred_opt):.4f}")

# Default threshold (0.5) cost
cost_default, _, _, _, _ = compute_business_cost(y_test, best_proba, 0.5, COST_FP, COST_FN)
print(f"\nCost at threshold=0.50: ${cost_default:,.0f}")
print(f"Cost reduction:         ${cost_default - opt_cost:,.0f} ({(cost_default - opt_cost)/cost_default:.1%})")

# ─────────────────────────────────────────────
# 5. MAIN FIGURE
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(22, 18), facecolor='#0f1117')
fig.suptitle('Task 4: Loan Default Risk — Binary Classification & Cost Optimization',
             fontsize=17, fontweight='bold', color='white', y=0.99)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
colors = {'Logistic Regression': '#4FC3F7', 'GradientBoosting': '#81C784'}

# ── ROC Curves ──
ax_roc = fig.add_subplot(gs[0, :2])
ax_roc.set_facecolor('#1a1d27')
ax_roc.plot([0,1],[0,1],'--', color='#555', lw=1)
for name, res in results.items():
    ax_roc.plot(res['fpr'], res['tpr'], color=colors[name], lw=2.5,
                label=f'{name} (AUC={res["auc"]:.4f})')
ax_roc.set_xlabel('False Positive Rate', color='white')
ax_roc.set_ylabel('True Positive Rate', color='white')
ax_roc.set_title('ROC Curves', color='white', fontsize=12)
ax_roc.legend(facecolor='#252836', labelcolor='white')
ax_roc.tick_params(colors='white')
for sp in ax_roc.spines.values(): sp.set_edgecolor('#333')
ax_roc.grid(alpha=0.12, color='white')

# ── Confusion Matrices ──
for i, (name, res) in enumerate(results.items()):
    ax_cm = fig.add_subplot(gs[0, 2] if i == 0 else gs[1, 0])
    ax_cm.set_facecolor('#1a1d27')
    y_pred_05 = (res['y_proba'] >= 0.5).astype(int)
    cm05 = confusion_matrix(y_test, y_pred_05)
    sns.heatmap(cm05, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                cbar=False, linewidths=1, linecolor='#1e2130')
    ax_cm.set_title(f'{name}\nThresh=0.5  F1={res["f1"]:.3f}  AUC={res["auc"]:.3f}',
                    color='white', fontsize=9)
    ax_cm.set_xlabel('Predicted', color='white', fontsize=9)
    ax_cm.set_ylabel('Actual', color='white', fontsize=9)
    ax_cm.set_xticklabels(['No Default','Default'], color='white', fontsize=8)
    ax_cm.set_yticklabels(['No Default','Default'], color='white', fontsize=8, rotation=0)
    ax_cm.tick_params(colors='white')
    for sp in ax_cm.spines.values(): sp.set_edgecolor('#333')
    ax_cm.set_facecolor('#1a1d27')

# ── Business Cost vs Threshold ──
ax_cost = fig.add_subplot(gs[1, 1:])
ax_cost.set_facecolor('#1a1d27')
ax_cost.plot(thresholds, [c/1e6 for c in costs], color='#FF6B6B', lw=2.5, label='Total Business Cost')
ax_cost.axvline(opt_threshold, color='#FFB74D', ls='--', lw=2,
                label=f'Optimal threshold={opt_threshold:.2f}')
ax_cost.axvline(0.5, color='#aaa', ls=':', lw=1.5, label='Default threshold=0.50')
ax_cost.scatter([opt_threshold], [opt_cost/1e6], s=120, color='#FFB74D', zorder=5)
ax_cost.set_xlabel('Classification Threshold', color='white')
ax_cost.set_ylabel('Total Business Cost ($M)', color='white')
ax_cost.set_title(f'Cost Optimization Curve\nFP Cost=${COST_FP:,}  |  FN Cost=${COST_FN:,}',
                  color='white', fontsize=11)
ax_cost.legend(facecolor='#252836', labelcolor='white')
ax_cost.tick_params(colors='white')
for sp in ax_cost.spines.values(): sp.set_edgecolor('#333')
ax_cost.grid(alpha=0.12, color='white')

ax_cost.annotate(f'Min Cost\n${opt_cost/1e6:.2f}M\n@t={opt_threshold:.2f}',
                 xy=(opt_threshold, opt_cost/1e6),
                 xytext=(opt_threshold + 0.12, opt_cost/1e6 + 0.5),
                 arrowprops=dict(arrowstyle='->', color='#FFB74D'),
                 color='#FFB74D', fontsize=9)

# ── Optimal Threshold CM ──
ax_cm_opt = fig.add_subplot(gs[2, 0])
ax_cm_opt.set_facecolor('#1a1d27')
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', ax=ax_cm_opt,
            cbar=False, linewidths=1, linecolor='#1e2130')
ax_cm_opt.set_title(f'GBR @ Optimal Threshold={opt_threshold:.2f}\nCost=${opt_cost:,.0f}  F1={f1_score(y_test, y_pred_opt):.3f}',
                    color='white', fontsize=10)
ax_cm_opt.set_xlabel('Predicted', color='white', fontsize=9)
ax_cm_opt.set_ylabel('Actual', color='white', fontsize=9)
ax_cm_opt.set_xticklabels(['No Default','Default'], color='white', fontsize=8)
ax_cm_opt.set_yticklabels(['No Default','Default'], color='white', fontsize=8, rotation=0)
ax_cm_opt.tick_params(colors='white')
for sp in ax_cm_opt.spines.values(): sp.set_edgecolor('#333')

# ── Feature Importance ──
ax_fi = fig.add_subplot(gs[2, 1:])
ax_fi.set_facecolor('#1a1d27')
gbr_model = results['GradientBoosting']['model']
fi        = gbr_model.feature_importances_
fi_df     = pd.DataFrame({'feature': feature_names, 'importance': fi})
fi_df     = fi_df.sort_values('importance', ascending=True).tail(15)
bar_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi_df)))
ax_fi.barh(range(len(fi_df)), fi_df['importance'], color=bar_colors, alpha=0.85)
ax_fi.set_yticks(range(len(fi_df)))
ax_fi.set_yticklabels(fi_df['feature'], color='white', fontsize=9)
ax_fi.set_title('GradientBoosting — Feature Importance (Top 15)', color='white', fontsize=11)
ax_fi.set_xlabel('Importance Score', color='white')
ax_fi.tick_params(colors='white')
for sp in ax_fi.spines.values(): sp.set_edgecolor('#333')
ax_fi.grid(axis='x', alpha=0.12, color='white')

# Add cost summary text box
ax_fi.text(0.98, 0.04,
           f"Cost Summary\n"
           f"─────────────────\n"
           f"FP Cost (Approve Defaulter): ${COST_FP:,}\n"
           f"FN Cost (Reject Good Client): ${COST_FN:,}\n"
           f"Default Threshold Cost: ${cost_default:,.0f}\n"
           f"Optimal Threshold Cost: ${opt_cost:,.0f}\n"
           f"Savings: ${cost_default - opt_cost:,.0f} ({(cost_default-opt_cost)/cost_default:.1%})",
           transform=ax_fi.transAxes, ha='right', va='bottom',
           fontsize=8.5, color='white',
           bbox=dict(boxstyle='round', facecolor='#252836', edgecolor='#FFB74D', alpha=0.9))

plt.savefig('./task4_loan_default.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("✅ Saved: task4_loan_default.png")
print("\n✅ Task 4 complete.")

Dataset shape: (10000, 20)
Default rate: 14.140%
Train: 8000  Test: 2000
Test default rate: 14.150%

Logistic Regression: AUC=0.7742  F1=0.4053

GradientBoosting: AUC=0.7560  F1=0.1793

OPTIMAL THRESHOLD: 0.852
MINIMUM COST:      $1,410,000
CM at optimal threshold:
[[1717    0]
 [ 282    1]]
F1 at optimal: 0.0070

Cost at threshold=0.50: $3,850,000
Cost reduction:         $2,440,000 (63.4%)
✅ Saved: task4_loan_default.png

✅ Task 4 complete.
